In [49]:
# Importing packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [50]:
# Importing dataset
raw = pd.read_csv('d1_all_match_results.csv')
raw

,season,date,event,is_dual_meet,weight_class,result,result_type,score,opponent,opponent_id,opponent_school,wrestler,wrestler_id,wrestler_school
0,2014,03/08,EIWA Championships,False,125,W,DEC,5 - 4,Caleb Richardson,12113,Pennsylvania,Chris McGinley,12050,Boston U
1,2014,03/08,EIWA Championships,False,125,L,FALL,2:48,David Terao,12068,American,Chris McGinley,12050,Boston U
2,2014,02/08,Princeton - Boston U Dual,True,125,W,MD,12 - 0,Ryan Cash,12045,Princeton,Chris McGinley,12050,Boston U
3,2014,02/02,Boston U - Binghamton Dual,True,125,L,DEC,9 - 6,David White,21963,Binghamton,Chris McGinley,12050,Boston U
4,2014,01/25,Harvard - Boston U Dual,True,125,W,MD,15 - 5,Max Mejia,17866,Harvard,Chris McGinley,12050,Boston U
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439444,2025,11/23,Utah Valley - NC State Dual,True,197,L,MD,14 - 3,Kael Bennie,84802,Utah Valley,Chase Horne,72952,Nc State
439445,2025,11/03,Battle at The Citadel,False,197,L,MFOR,0 - 0,Vincent Lee,93063,Duke,Chase Horne,72952,Nc State
439446,2025,11/03,Battle at The Citadel,False,197,W,DEC,8 - 3,Kwasi Bonsu,84783,Duke,Chase Horne,72952,Nc State
439447,2025,11/03,Battle at The Citadel,False,197,W,MD,15 - 3,Gunnar Pool,82480,Appalachian State,Chase Horne,72952,Nc State


In [51]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439449 entries, 0 to 439448
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   season           439449 non-null  int64 
 1   date             439449 non-null  object
 2   event            439449 non-null  object
 3   is_dual_meet     439449 non-null  bool  
 4   weight_class     439449 non-null  int64 
 5   result           439449 non-null  object
 6   result_type      439449 non-null  object
 7   score            439449 non-null  object
 8   opponent         439449 non-null  object
 9   opponent_id      439449 non-null  int64 
 10  opponent_school  439449 non-null  object
 11  wrestler         439449 non-null  object
 12  wrestler_id      439449 non-null  int64 
 13  wrestler_school  439449 non-null  object
dtypes: bool(1), int64(4), object(9)
memory usage: 44.0+ MB


In [52]:
# Checking for any duplicated matches
len(raw[raw.duplicated(keep=False)])

0

In [53]:
# Adding datetime column that captures full date
raw['temp_date'] = pd.to_datetime(raw['season'].astype('str') + '/' + raw['date'], format='%Y/%m/%d')
    
# Extract month to identify which dates need adjustment
raw['month'] = raw['temp_date'].dt.month

# Adjust year for months that are typically in the previous calendar year
# Assuming August-December (months 8-12) belong to the previous calendar year
raw['datetime'] = raw.apply(lambda row: 
    row['temp_date'].replace(year=int(row['season']) - 1) 
    if row['month'] >= 8  # August through December
    else row['temp_date'], axis=1)

# Clean up temporary columns and move datetime column
raw = raw.drop(['temp_date', 'month', 'date'], axis=1)
datetime = raw.pop('datetime')
raw.insert(1, 'date', datetime)

In [54]:
# Sorting the data by date and wrestler ID
raw = raw.sort_values(['date', 'wrestler_id']).copy()
raw['date']

7342     2013-11-01
7212     2013-11-01
13680    2013-11-01
17930    2013-11-01
17956    2013-11-01
            ...    
402469   2025-03-20
423840   2025-03-20
423841   2025-03-20
423842   2025-03-20
423843   2025-03-20
Name: date, Length: 439449, dtype: datetime64[ns]

In [55]:
# Editing Season column to reflect full span
raw['season'] = (raw['season'] - 1).astype('str') + '/' + raw['season'].astype('str')
raw['season']

7342      2013/2014
7212      2013/2014
13680     2013/2014
17930     2013/2014
17956     2013/2014
            ...    
402469    2024/2025
423840    2024/2025
423841    2024/2025
423842    2024/2025
423843    2024/2025
Name: season, Length: 439449, dtype: object

In [56]:
raw = raw.reset_index(drop=True)
raw

,season,date,event,is_dual_meet,weight_class,result,result_type,score,opponent,opponent_id,opponent_school,wrestler,wrestler_id,wrestler_school
0,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,165,L,DEC,6 - 1,Steven Monk,37120,North Dakota State,Jordan Gagliano,11477,Missouri
1,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,141,L,DEC,4 - 1,Justin LaValle,37157,North Dakota State,Trevor Jauch,11593,Missouri
2,2013/2014,2013-11-01,Oklahoma Gold Classic,False,125,W,DEC,12 - 6,Max Soria,12190,Buffalo,David Terao,12068,American
3,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,174,W,DEC,3 - 1,Michael England,20057,Missouri,Hayden Zillmer,12280,North Dakota State
4,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,184,L,DEC,5 - 2,Johnny Eblen,12537,Missouri,Kurtis Julson,12375,North Dakota State
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439444,2024/2025,2025-03-20,NCAA Championships,False,174,L,FALL,2:46,Levi Haines,72657,Penn State,Branson John,92961,Maryland
439445,2024/2025,2025-03-20,NCAA Championships,False,184,L,TF5,19 - 4 2:41,Donnell Washington,57738,Indiana,Eddie Neitenbach,92977,Wyoming
439446,2024/2025,2025-03-20,NCAA Championships,False,184,W,SV-1,9 - 6,Nick Fine,71734,Columbia,Eddie Neitenbach,92977,Wyoming
439447,2024/2025,2025-03-20,NCAA Championships,False,184,W,FALL,1:08,TJ McDonnell,79786,Oregon State,Eddie Neitenbach,92977,Wyoming


In [57]:
# Adding unique match ID
def generate_match_id(row):
    import hashlib
    key = f"{row['season']}_{row['date']}_{row['wrestler_id']}_{row['opponent_id']}_{row['weight_class']}"
    return hashlib.md5(key.encode()).hexdigest()

In [58]:
raw['match_id'] = raw.apply(generate_match_id, axis=1)
raw

,season,date,event,is_dual_meet,weight_class,result,result_type,score,opponent,opponent_id,opponent_school,wrestler,wrestler_id,wrestler_school,match_id
0,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,165,L,DEC,6 - 1,Steven Monk,37120,North Dakota State,Jordan Gagliano,11477,Missouri,dca00b7d69ab50e1eebd9f8559cefd70
1,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,141,L,DEC,4 - 1,Justin LaValle,37157,North Dakota State,Trevor Jauch,11593,Missouri,87a87074a5aa48b024b9417ca68c41e7
2,2013/2014,2013-11-01,Oklahoma Gold Classic,False,125,W,DEC,12 - 6,Max Soria,12190,Buffalo,David Terao,12068,American,9845500d49d71701f7d5f21ef9a49886
3,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,174,W,DEC,3 - 1,Michael England,20057,Missouri,Hayden Zillmer,12280,North Dakota State,7aa47adcb56c9b4ef13b6465ab93f6c5
4,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,184,L,DEC,5 - 2,Johnny Eblen,12537,Missouri,Kurtis Julson,12375,North Dakota State,d18ad995c421dd3971099e243f984089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439444,2024/2025,2025-03-20,NCAA Championships,False,174,L,FALL,2:46,Levi Haines,72657,Penn State,Branson John,92961,Maryland,6dff9e3cd5dc51c1f0a7c3fa5ff2f0f3
439445,2024/2025,2025-03-20,NCAA Championships,False,184,L,TF5,19 - 4 2:41,Donnell Washington,57738,Indiana,Eddie Neitenbach,92977,Wyoming,63e5df7db75246d1392df070f65d1190
439446,2024/2025,2025-03-20,NCAA Championships,False,184,W,SV-1,9 - 6,Nick Fine,71734,Columbia,Eddie Neitenbach,92977,Wyoming,2074c461dd856b6431df06c36f2db16c
439447,2024/2025,2025-03-20,NCAA Championships,False,184,W,FALL,1:08,TJ McDonnell,79786,Oregon State,Eddie Neitenbach,92977,Wyoming,51a9553f53785e2f7a726b7e57450c38


In [59]:
raw.result_type.value_counts()

result_type
DEC     215259
FALL     73984
MD       73850
TF5      37792
SV-1     15152
MFOR     11262
INJ       3495
TB-1      2545
TF4       2085
TB-2      1812
CMFF       795
SV-2       569
DQ         414
DEF        184
TB-3       136
SV-3        99
FOR         16
Name: count, dtype: int64

In [60]:
# Seems to be a decent number of matches with result types like 'MFOR', 'INJ', 'CMFF', 'DQ', 'DEF', and 'FOR'
print(len(raw))
print(len(raw[raw['result_type'].isin(['MFOR', 'INJ', 'CMFF', 'DQ', 'DEF', 'FOR'])]))

439449
16166


In [61]:
# Removing any matches that were not actually wrestled or were cut short (forfeit, injury, etc.)
raw = raw[~raw['result_type'].isin(['MFOR', 'INJ', 'CMFF', 'DQ', 'DEF', 'FOR'])]
len(raw)

423283

In [62]:
raw.result_type.value_counts()

result_type
DEC     215259
FALL     73984
MD       73850
TF5      37792
SV-1     15152
TB-1      2545
TF4       2085
TB-2      1812
SV-2       569
TB-3       136
SV-3        99
Name: count, dtype: int64

In [63]:
raw.score.value_counts().tail(20)

score
24 - 6 4:46    1
23 - 6 6:18    1
23 - 7 5:30    1
22 - 5 6:34    1
19 - 3 1:54    1
18 - 1 2:11    1
17 - 1 2:30    1
19 - 2 3:36    1
17 - 2 2:56    1
18 - 3 4:01    1
20 - 2 2:09    1
16 - 1 3:13    1
24 - 5 4:01    1
21 - 4 5:08    1
18 - 3 1:47    1
20 - 3 2:57    1
17 - 1 4:49    1
18 - 3 5:52    1
22 - 6 1:47    1
22 - 6 6:03    1
Name: count, dtype: int64

In [64]:
# looks like all of the technical falls have scores and times attached
raw['match_duration'] = raw['score'].str.extract(r'(\d{1,2}:\d{2})')[0].fillna('7:00')
raw

C:\Users\kikip\AppData\Local\Temp\ipykernel_13672\1957134846.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw['match_duration'] = raw['score'].str.extract(r'(\d{1,2}:\d{2})')[0].fillna('7:00')


,season,date,event,is_dual_meet,weight_class,result,result_type,score,opponent,opponent_id,opponent_school,wrestler,wrestler_id,wrestler_school,match_id,match_duration
0,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,165,L,DEC,6 - 1,Steven Monk,37120,North Dakota State,Jordan Gagliano,11477,Missouri,dca00b7d69ab50e1eebd9f8559cefd70,7:00
1,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,141,L,DEC,4 - 1,Justin LaValle,37157,North Dakota State,Trevor Jauch,11593,Missouri,87a87074a5aa48b024b9417ca68c41e7,7:00
2,2013/2014,2013-11-01,Oklahoma Gold Classic,False,125,W,DEC,12 - 6,Max Soria,12190,Buffalo,David Terao,12068,American,9845500d49d71701f7d5f21ef9a49886,7:00
3,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,174,W,DEC,3 - 1,Michael England,20057,Missouri,Hayden Zillmer,12280,North Dakota State,7aa47adcb56c9b4ef13b6465ab93f6c5,7:00
4,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,184,L,DEC,5 - 2,Johnny Eblen,12537,Missouri,Kurtis Julson,12375,North Dakota State,d18ad995c421dd3971099e243f984089,7:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439444,2024/2025,2025-03-20,NCAA Championships,False,174,L,FALL,2:46,Levi Haines,72657,Penn State,Branson John,92961,Maryland,6dff9e3cd5dc51c1f0a7c3fa5ff2f0f3,2:46
439445,2024/2025,2025-03-20,NCAA Championships,False,184,L,TF5,19 - 4 2:41,Donnell Washington,57738,Indiana,Eddie Neitenbach,92977,Wyoming,63e5df7db75246d1392df070f65d1190,2:41
439446,2024/2025,2025-03-20,NCAA Championships,False,184,W,SV-1,9 - 6,Nick Fine,71734,Columbia,Eddie Neitenbach,92977,Wyoming,2074c461dd856b6431df06c36f2db16c,7:00
439447,2024/2025,2025-03-20,NCAA Championships,False,184,W,FALL,1:08,TJ McDonnell,79786,Oregon State,Eddie Neitenbach,92977,Wyoming,51a9553f53785e2f7a726b7e57450c38,1:08


In [65]:
if raw['result_type'].str.contains('TF').any():
    raw['score'] = raw['score'].replace(r' \d{1,2}:\d{2}', '', regex=True)
raw.score.value_counts()

C:\Users\kikip\AppData\Local\Temp\ipykernel_13672\2524348589.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw['score'] = raw['score'].replace(r' \d{1,2}:\d{2}', '', regex=True)


score
3 - 2      12430
3 - 1      11529
4 - 2       9991
5 - 3       8496
4 - 1       8467
           ...  
23 - 18        1
0:06           1
9:14           1
30 - 15        1
8:32           1
Name: count, Length: 878, dtype: int64

In [66]:
raw[['score1', 'score2']] = raw['score'].str.split(' - ', expand=True)

raw['wrestler_score'] = np.where(raw['result'] == 'W', raw['score1'], 
                       np.where(raw['result'] == 'L', raw['score2'], 0))
raw['opponent_score'] = np.where(raw['result'] == 'W', raw['score2'], 
                       np.where(raw['result'] == 'L', raw['score1'], 0))

raw = raw.drop(['score1', 'score2'], axis=1)
raw

C:\Users\kikip\AppData\Local\Temp\ipykernel_13672\2178407512.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw[['score1', 'score2']] = raw['score'].str.split(' - ', expand=True)
C:\Users\kikip\AppData\Local\Temp\ipykernel_13672\2178407512.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw[['score1', 'score2']] = raw['score'].str.split(' - ', expand=True)
C:\Users\kikip\AppData\Local\Temp\ipykernel_13672\2178407512.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

,season,date,event,is_dual_meet,weight_class,result,result_type,score,opponent,opponent_id,opponent_school,wrestler,wrestler_id,wrestler_school,match_id,match_duration,wrestler_score,opponent_score
0,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,165,L,DEC,6 - 1,Steven Monk,37120,North Dakota State,Jordan Gagliano,11477,Missouri,dca00b7d69ab50e1eebd9f8559cefd70,7:00,1,6
1,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,141,L,DEC,4 - 1,Justin LaValle,37157,North Dakota State,Trevor Jauch,11593,Missouri,87a87074a5aa48b024b9417ca68c41e7,7:00,1,4
2,2013/2014,2013-11-01,Oklahoma Gold Classic,False,125,W,DEC,12 - 6,Max Soria,12190,Buffalo,David Terao,12068,American,9845500d49d71701f7d5f21ef9a49886,7:00,12,6
3,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,174,W,DEC,3 - 1,Michael England,20057,Missouri,Hayden Zillmer,12280,North Dakota State,7aa47adcb56c9b4ef13b6465ab93f6c5,7:00,3,1
4,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,184,L,DEC,5 - 2,Johnny Eblen,12537,Missouri,Kurtis Julson,12375,North Dakota State,d18ad995c421dd3971099e243f984089,7:00,2,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439444,2024/2025,2025-03-20,NCAA Championships,False,174,L,FALL,2:46,Levi Haines,72657,Penn State,Branson John,92961,Maryland,6dff9e3cd5dc51c1f0a7c3fa5ff2f0f3,2:46,None,2:46
439445,2024/2025,2025-03-20,NCAA Championships,False,184,L,TF5,19 - 4,Donnell Washington,57738,Indiana,Eddie Neitenbach,92977,Wyoming,63e5df7db75246d1392df070f65d1190,2:41,4,19
439446,2024/2025,2025-03-20,NCAA Championships,False,184,W,SV-1,9 - 6,Nick Fine,71734,Columbia,Eddie Neitenbach,92977,Wyoming,2074c461dd856b6431df06c36f2db16c,7:00,9,6
439447,2024/2025,2025-03-20,NCAA Championships,False,184,W,FALL,1:08,TJ McDonnell,79786,Oregon State,Eddie Neitenbach,92977,Wyoming,51a9553f53785e2f7a726b7e57450c38,1:08,1:08,None


In [67]:
raw['wrestler_score'] = np.where(raw['result_type'] != 'FALL', raw['wrestler_score'], 0)
raw['opponent_score'] = np.where(raw['result_type'] != 'FALL', raw['opponent_score'], 0)
raw['score'] = np.where(raw['result_type'] != 'FALL', raw['score'], 'FALL')
raw

,season,date,event,is_dual_meet,weight_class,result,result_type,score,opponent,opponent_id,opponent_school,wrestler,wrestler_id,wrestler_school,match_id,match_duration,wrestler_score,opponent_score
0,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,165,L,DEC,6 - 1,Steven Monk,37120,North Dakota State,Jordan Gagliano,11477,Missouri,dca00b7d69ab50e1eebd9f8559cefd70,7:00,1,6
1,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,141,L,DEC,4 - 1,Justin LaValle,37157,North Dakota State,Trevor Jauch,11593,Missouri,87a87074a5aa48b024b9417ca68c41e7,7:00,1,4
2,2013/2014,2013-11-01,Oklahoma Gold Classic,False,125,W,DEC,12 - 6,Max Soria,12190,Buffalo,David Terao,12068,American,9845500d49d71701f7d5f21ef9a49886,7:00,12,6
3,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,174,W,DEC,3 - 1,Michael England,20057,Missouri,Hayden Zillmer,12280,North Dakota State,7aa47adcb56c9b4ef13b6465ab93f6c5,7:00,3,1
4,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,184,L,DEC,5 - 2,Johnny Eblen,12537,Missouri,Kurtis Julson,12375,North Dakota State,d18ad995c421dd3971099e243f984089,7:00,2,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439444,2024/2025,2025-03-20,NCAA Championships,False,174,L,FALL,FALL,Levi Haines,72657,Penn State,Branson John,92961,Maryland,6dff9e3cd5dc51c1f0a7c3fa5ff2f0f3,2:46,0,0
439445,2024/2025,2025-03-20,NCAA Championships,False,184,L,TF5,19 - 4,Donnell Washington,57738,Indiana,Eddie Neitenbach,92977,Wyoming,63e5df7db75246d1392df070f65d1190,2:41,4,19
439446,2024/2025,2025-03-20,NCAA Championships,False,184,W,SV-1,9 - 6,Nick Fine,71734,Columbia,Eddie Neitenbach,92977,Wyoming,2074c461dd856b6431df06c36f2db16c,7:00,9,6
439447,2024/2025,2025-03-20,NCAA Championships,False,184,W,FALL,FALL,TJ McDonnell,79786,Oregon State,Eddie Neitenbach,92977,Wyoming,51a9553f53785e2f7a726b7e57450c38,1:08,0,0


In [ ]:
# all below use for feature engineering
raw = raw.drop(['score'], axis=1)
raw

,season,date,event,is_dual_meet,weight_class,result,result_type,opponent,opponent_id,opponent_school,wrestler,wrestler_id,wrestler_school,match_id,match_duration,wrestler_score,opponent_score
0,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,165,L,DEC,Steven Monk,37120,North Dakota State,Jordan Gagliano,11477,Missouri,dca00b7d69ab50e1eebd9f8559cefd70,7:00,1,6
1,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,141,L,DEC,Justin LaValle,37157,North Dakota State,Trevor Jauch,11593,Missouri,87a87074a5aa48b024b9417ca68c41e7,7:00,1,4
2,2013/2014,2013-11-01,Oklahoma Gold Classic,False,125,W,DEC,Max Soria,12190,Buffalo,David Terao,12068,American,9845500d49d71701f7d5f21ef9a49886,7:00,12,6
3,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,174,W,DEC,Michael England,20057,Missouri,Hayden Zillmer,12280,North Dakota State,7aa47adcb56c9b4ef13b6465ab93f6c5,7:00,3,1
4,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,184,L,DEC,Johnny Eblen,12537,Missouri,Kurtis Julson,12375,North Dakota State,d18ad995c421dd3971099e243f984089,7:00,2,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439444,2024/2025,2025-03-20,NCAA Championships,False,174,L,FALL,Levi Haines,72657,Penn State,Branson John,92961,Maryland,6dff9e3cd5dc51c1f0a7c3fa5ff2f0f3,2:46,0,0
439445,2024/2025,2025-03-20,NCAA Championships,False,184,L,TF5,Donnell Washington,57738,Indiana,Eddie Neitenbach,92977,Wyoming,63e5df7db75246d1392df070f65d1190,2:41,4,19
439446,2024/2025,2025-03-20,NCAA Championships,False,184,W,SV-1,Nick Fine,71734,Columbia,Eddie Neitenbach,92977,Wyoming,2074c461dd856b6431df06c36f2db16c,7:00,9,6
439447,2024/2025,2025-03-20,NCAA Championships,False,184,W,FALL,TJ McDonnell,79786,Oregon State,Eddie Neitenbach,92977,Wyoming,51a9553f53785e2f7a726b7e57450c38,1:08,0,0


In [69]:
# Reordering columns
raw = raw.iloc[:, [13, 0, 1, 2, 3, 4, 10, 11, 12, 7, 8, 9, 5, 6, 15, 16, 14]]
raw

,match_id,season,date,event,is_dual_meet,weight_class,wrestler,wrestler_id,wrestler_school,opponent,opponent_id,opponent_school,result,result_type,wrestler_score,opponent_score,match_duration
0,dca00b7d69ab50e1eebd9f8559cefd70,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,165,Jordan Gagliano,11477,Missouri,Steven Monk,37120,North Dakota State,L,DEC,1,6,7:00
1,87a87074a5aa48b024b9417ca68c41e7,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,141,Trevor Jauch,11593,Missouri,Justin LaValle,37157,North Dakota State,L,DEC,1,4,7:00
2,9845500d49d71701f7d5f21ef9a49886,2013/2014,2013-11-01,Oklahoma Gold Classic,False,125,David Terao,12068,American,Max Soria,12190,Buffalo,W,DEC,12,6,7:00
3,7aa47adcb56c9b4ef13b6465ab93f6c5,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,174,Hayden Zillmer,12280,North Dakota State,Michael England,20057,Missouri,W,DEC,3,1,7:00
4,d18ad995c421dd3971099e243f984089,2013/2014,2013-11-01,Missouri - North Dakota State Dual,True,184,Kurtis Julson,12375,North Dakota State,Johnny Eblen,12537,Missouri,L,DEC,2,5,7:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439444,6dff9e3cd5dc51c1f0a7c3fa5ff2f0f3,2024/2025,2025-03-20,NCAA Championships,False,174,Branson John,92961,Maryland,Levi Haines,72657,Penn State,L,FALL,0,0,2:46
439445,63e5df7db75246d1392df070f65d1190,2024/2025,2025-03-20,NCAA Championships,False,184,Eddie Neitenbach,92977,Wyoming,Donnell Washington,57738,Indiana,L,TF5,4,19,2:41
439446,2074c461dd856b6431df06c36f2db16c,2024/2025,2025-03-20,NCAA Championships,False,184,Eddie Neitenbach,92977,Wyoming,Nick Fine,71734,Columbia,W,SV-1,9,6,7:00
439447,51a9553f53785e2f7a726b7e57450c38,2024/2025,2025-03-20,NCAA Championships,False,184,Eddie Neitenbach,92977,Wyoming,TJ McDonnell,79786,Oregon State,W,FALL,0,0,1:08
